# Котейка читает: «Trustworthy Online Controlled Experiments»

Ноутбук к разбору книги Кохави, Тан и Сюй (2020): https://dilyakhakimova.com/books/kohavi-trustworthy/

Три проверки доверия к A/B-тесту: SRM χ²-критерием, симуляция подглядывания и A/A-стенд на 1000 прогонов. Код и числа наши, по описанию методов; нужны только Python и numpy. Подставь свои числа и запусти.

## Проверка SRM χ²-критерием

Поменяй n_a и n_b на числа из своего теста. Если сплит не 50/50, задай share_a.

In [ ]:
import numpy as np
from math import erfc, sqrt

def srm_check(n_a, n_b, share_a=0.5):
    """χ²-критерий: совпадает ли соотношение групп с задуманным сплитом."""
    total = n_a + n_b
    exp_a, exp_b = total * share_a, total * (1 - share_a)
    chi2 = (n_a - exp_a) ** 2 / exp_a + (n_b - exp_b) ** 2 / exp_b
    p = erfc(sqrt(chi2 / 2))   # p-value для χ² с одной степенью свободы
    return chi2, p

# кафе: сплит 50/50, в группе B антифрод отрезал часть гостей
n_a, n_b = 500_000, 493_000
chi2, p = srm_check(n_a, n_b)
print(f"A = {n_a}, B = {n_b}, доля A = {n_a / (n_a + n_b):.4f}")
print(f"χ² = {chi2:.1f}, p-value = {p:.2g}")
print("SRM: отчёту не верим, ищем причину" if p < 0.0005 else "перекос в пределах шума")

## Симуляция подглядывания: 1000 A/A-тестов

Эффекта нет ни в одном тесте. Сравни, сколько «побед» дают проверка в конце и проверка каждый день. Поменяй days — посмотри, как растут ложные победы.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
tests, days, per_day, conv = 1000, 20, 500, 0.10   # 1000 A/A-тестов по 20 дней, по 500 гостей в день в группе

# накопленные заказы в группах A и B; эффекта нет: конверсия одна и та же
a = rng.binomial(per_day, conv, (tests, days)).cumsum(axis=1)
b = rng.binomial(per_day, conv, (tests, days)).cumsum(axis=1)
n = per_day * np.arange(1, days + 1)

pool = (a + b) / (2 * n)
se = np.sqrt(pool * (1 - pool) * 2 / n)
z = np.abs(b - a) / n / se
significant = z > 1.96          # «значимо» при α = 0,05

print(f"смотрим один раз, в конце:          {significant[:, -1].mean():.1%} ложных побед")
print(f"смотрим каждый день, стоп при «значимо»: {significant.any(axis=1).mean():.1%} ложных побед")

## A/A-стенд: 1000 прогонов и гистограмма p-value

Честная платформа даёт ровную гистограмму. Испорть её: поставь bad_variance = True, как будто дисперсию посчитали по заказам, а не по гостям.

In [ ]:
import numpy as np
from math import erfc, sqrt

rng = np.random.default_rng(7)
runs, n, conv = 1000, 5000, 0.10   # 1000 A/A-тестов, по 5000 гостей в группе, конверсия 10 %
bad_variance = False               # True — дисперсию недооценили (например, считали по заказам, а не по гостям)

a = rng.binomial(n, conv, runs)
b = rng.binomial(n, conv, runs)
pool = (a + b) / (2 * n)
se = np.sqrt(pool * (1 - pool) * 2 / n)
if bad_variance:
    se = se * 0.7
z = np.abs(b - a) / n / se
pvals = np.array([erfc(x / sqrt(2)) for x in z])

print(f"значимых при α = 0,05: {np.mean(pvals < 0.05):.1%} (у честной платформы — около 5 %)")
hist, _ = np.histogram(pvals, bins=10, range=(0, 1))
for i, h in enumerate(hist):
    print(f"p {i / 10:.1f}–{(i + 1) / 10:.1f} {'█' * (h // 5)} {h}")